# 청킹과 임베딩 — 기초 (한글 소설)

문서 하나, 청킹 방법 세 가지, 그리고 **검색했을 때 실제로 무엇이 돌아오는지** 눈으로 확인합니다.
벤치마크도, 정답 라벨도, 점수표도 없습니다. 직접 읽고 판단하는 것이 목적입니다.

문서는 현진건의 단편소설 **「운수 좋은 날」(1924)** 원문을 한국어 위키문헌에서 내려받아 사용합니다.
저작권이 만료된 작품이고 중학교 교과서에도 실려 있어, **여러분이 이미 아는 이야기**입니다.
그래서 *"아내는 어떻게 되었나?"* 라고 물었을 때 돌아온 대목이 맞는지 **읽어 보면 바로 판단할 수 있습니다.**

> 같은 코드를 백과사전 문서(한국어 위키백과 「인공지능」)로 돌린 버전은 `chunk_basic_ko1.ipynb`에 있습니다.
> 두 노트북을 비교하면 **글의 종류에 따라 청킹과 검색이 어떻게 달라지는지** 볼 수 있습니다.

문서 전체가 약 11,000토큰이라 임베딩 비용은 한 번에 **약 $0.0002**입니다.
청크 크기를 바꿔 가며 몇 번을 실험해도 1센트가 안 됩니다.

## 1. 문서 가져오기

데이터 준비는 이 셀이 전부입니다. 별도의 데이터셋 라이브러리도, 스키마도 없습니다.

**위키문헌(Wikisource)** 은 저작권이 만료된 원문을 모아 둔 위키미디어 프로젝트입니다.
위키백과와 **API 주소 형식이 같아서**, 도메인만 `ko.wikisource.org`로 바꾸면 됩니다.
`explaintext=1`을 주면 `{{머리말}}` 같은 위키 문법이 제거된 **순수 텍스트**를 받을 수 있습니다.

> ⚠️ `action=raw` 방식(`index.php?title=...&action=raw`)도 있지만, 그쪽은 JSON이 아니라
> 위키 문법 원문을 돌려줍니다. `resp.json()`을 호출하면 `JSONDecodeError`가 납니다.
> 아래처럼 **`api.php`의 `extracts`** 를 쓰는 편이 간단합니다.

In [1]:
import re

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

In [2]:
TITLE = "운수 좋은 날"

resp = requests.get(
    "https://ko.wikisource.org/w/api.php",   # 위키백과가 아니라 위키문헌(원문 저장소)
    params={
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,     # 위키 문법을 제거한 순수 텍스트로 받기
        "redirects": 1,
        "titles": TITLE,
    },
    headers={"User-Agent": "llm-edu-notebook/1.0"},
    timeout=30,
)

pages = resp.json()["query"]["pages"]
doc = next(iter(pages.values()))["extract"].strip()

# 내려받은 원문 끝에는 위키문헌이 붙인 '== 라이선스 ==' 절이 딸려 온다.
# 작품 내용이 아니므로 잘라낸다. (실무에서도 머리말·꼬리말 제거는 늘 필요한 단계)
doc = re.split(r"^== 라이선스 ==$", doc, flags=re.M)[0].strip()

print(f"{len(doc):,}자   {ntok(doc):,}토큰")
print(f"절(==) {len(re.findall(r'^== .+ ==$', doc, re.M))}개")
print(f"줄 {len(doc.splitlines())}개 / 빈 줄로 나눈 덩어리 {len(re.split(r'\n\s*\n', doc))}개")
print()
print(doc[:400])

10,284자   10,998토큰
절(==) 0개
줄 90개 / 빈 줄로 나눈 덩어리 4개

🙝🙟
새침하게 흐린 품이 눈이 올 듯하더니 눈은 아니 오고 얼다가 만 비가 추적추적 내리었다.
이날이야말로 동소문 안에서 인력거꾼 노릇을 하는 김 첨지에게는 오래간만에도 닥친 운수 좋은 날이었다. 문안에(거기도 문밖은 아니지만) 들어간답시는 앞집 마나님을 전찻길까지 모셔다 드린 것을 비롯으로 행여나 손님이 있을까 하고 정류장에서 어정어정하며 내리는 사람 하나하나에게 거의 비는 듯한 눈결을 보내고 있다가 마침내 교원인 듯한 양복장이를 동광학교(東光學校)까지 태워다 주기로 되었다.
첫번에 삼십 전, 둘째 번에 오십 전 - 아침 댓바람에 그리 흔치 않은 일이었다. 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김 첨지는 십 전짜리 백통화 서 푼, 또는 다섯 푼이 찰깍하고 손바닥에 떨어질 제 거의 눈물을 


### 잠깐 — 한글은 토큰을 얼마나 쓸까?

| | 글자 수 | 토큰 수 | 글자당 토큰 |
|---|---|---|---|
| 영어 (이상한 나라의 앨리스) | 144,599자 | 36,926토큰 | 약 0.26 |
| **한글 (운수 좋은 날)** | 약 10,300자 | 약 11,000토큰 | **약 1.07** |

**한글은 글자 수보다 토큰 수가 오히려 많습니다.**
`cl100k_base` 토크나이저가 영어 위주로 만들어져서, 한글은 글자 하나가 토큰 1개 이상을 쓰는 일이 많기 때문입니다.
영어는 글자 4개가 토큰 1개쯤 되니, 같은 글자 수라면 **한글이 토큰을 약 4배** 씁니다.

실무에서 중요한 점은 두 가지입니다.

1. 영어 예제에서 본 `chunk_size` 숫자를 한글에 그대로 쓰면 **글자 기준으로는 훨씬 짧은 내용**이 담깁니다.
2. 같은 분량의 문서라도 한글은 **비용과 컨텍스트를 더 많이 씁니다.**

토큰에 대한 자세한 내용은 `../llm_intro/1_5_tokenizer.ipynb`를 참고하세요.

## 2. 왜 잘라야 할까?

이유는 두 가지이고, 둘 다 구체적입니다.

1. **임베딩 모델에 입력 한계가 있습니다** — `text-embedding-3-small`은 8,192토큰까지만 받습니다.
   이 소설은 약 11,000토큰이라 **단편소설 한 편도 한 번에 넣을 수 없습니다.**
2. **들어간다고 해도 작품 전체를 벡터 하나로 만들면 쓸모가 없습니다.**
   임베딩은 텍스트 전체의 평균 같은 것입니다. 소설 전체를 벡터 하나로 만들면
   "가난한 인력거꾼 이야기"가 되어 버려서, 어떤 장면을 물어도 비슷하게 가깝고
   **어떤 장면도 콕 집어내지 못합니다.**

청킹은 문서를 **각 벡터가 한 장면만 담을 만큼** 작게 나누는 작업입니다.

In [3]:
# 문서 전체를 임베딩하면 어떻게 될까? 직접 시도해서 거절당하는 것을 확인합니다.
try:
    ai.embeddings.create(model="text-embedding-3-small", input=[doc])
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:200])

BadRequestError
Error code: 400 - {'error': {'message': "Invalid 'input[0]': maximum input length is 8192 tokens.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


## 3. 자르는 방법 세 가지

아래 함수들은 모두 **텍스트를 받아 문자열 리스트를 돌려줄 뿐**입니다.
청커(chunker)란 그게 전부입니다.

| 방법 | 자르는 기준 | 특징 |
|---|---|---|
| `chunk_fixed` | 토큰 개수 | 일정한 크기. 문장 중간에서 잘림 |
| `chunk_paragraphs` | 줄 바꿈(문단) | 문장을 자르지 않음. 크기가 들쭉날쭉 |
| `chunk_sections` | `== 절 제목 ==` | **이 소설에는 절 제목이 없어서 무용지물** |

> **글의 종류가 청커를 고릅니다.** `chunk_sections`는 위키백과 문서(`chunk_basic_ko1.ipynb`)에서는
> 14개 절로 잘 나뉘었지만, 소설에는 `== 제목 ==`이 없어서 **작품 전체가 청크 1개**가 됩니다.
> 문서 구조에 맞는 청커를 골라야 한다는 것을 보여 주는 예입니다.

> **문단 구분자 주의.** 위키백과 문서는 문단 사이가 **빈 줄**이지만,
> 위키문헌 소설 원문은 **한 줄 바꿈**입니다. (위 셀에서 "빈 줄로 나눈 덩어리 5개"를 확인했습니다.)
> 그래서 아래 `chunk_paragraphs`는 `\n\s*\n`(빈 줄)이 아니라 `\n+`(한 번 이상의 줄 바꿈)로 나눕니다.
> 빈 줄만 찾으면 청크가 5개밖에 안 나와서 청킹이 사실상 동작하지 않습니다.

In [4]:
def chunk_fixed(text, size=200, overlap=0):
    """`size` 토큰마다 무조건 자른다. 문장이 어디서 끝나는지는 신경 쓰지 않는다."""
    ids = ENC.encode(text)
    step = size - overlap
    out = []
    for i in range(0, len(ids), step):
        piece = ids[i : i + size]
        if not piece:
            break
        out.append(ENC.decode(piece))
        if i + size >= len(ids):
            break
    return out


def chunk_paragraphs(text, target=200):
    """문단을 통째로 모아서 대략 `target` 토큰이 될 때까지 합친다."""
    paras = [p.strip() for p in re.split(r"\n+", text) if p.strip()]   # \n+ : 빈 줄이든 한 줄 바꿈이든 문단으로 인정
    out, buf, n = [], [], 0
    for p in paras:
        t = ntok(p)
        if buf and n + t > target:
            out.append("\n\n".join(buf))
            buf, n = [], 0
        buf.append(p)
        n += t
    if buf:
        out.append("\n\n".join(buf))
    return out


def chunk_sections(text):
    """'== 제목 ==' 절 단위로 자른다. 자연스럽지만 검색에 쓰기에는 너무 크다."""
    parts = re.split(r"(?=^== .+ ==$)", text, flags=re.M)
    return [p.strip() for p in parts if p.strip()]

In [5]:
for name, chunks in [
    ("fixed-200",       chunk_fixed(doc, 200)),
    ("fixed-200/ov50",  chunk_fixed(doc, 200, 50)),
    ("paragraphs-200",  chunk_paragraphs(doc, 200)),
    ("sections",        chunk_sections(doc)),
]:
    sizes = [ntok(c) for c in chunks]
    print(f"{name:18s} {len(chunks):>4}개 청크   "
          f"중앙값 {int(np.median(sizes)):>4}토큰   최대 {max(sizes):>5}토큰")

fixed-200            55개 청크   중앙값  200토큰   최대   200토큰
fixed-200/ov50       73개 청크   중앙값  200토큰   최대   200토큰
paragraphs-200       57개 청크   중앙값  190토큰   최대   345토큰
sections              1개 청크   중앙값 10998토큰   최대 10998토큰


## 4. 잘린 자리를 들여다보기

여기가 가장 눈여겨볼 부분입니다.
고정 크기 청킹은 토큰 수만 세다가 자르기 때문에, **문장 중간이나 심하면 단어 중간**에서 끊깁니다.

In [6]:
fixed = chunk_fixed(doc, 200)

print("=" * 72)
print("청크 5의 끝부분:")
print("=" * 72)
print("..." + fixed[5][-200:])
print()
print("=" * 72)
print("청크 6의 시작부분  <- 위 문장이 여기서 이어진다")
print("=" * 72)
print(fixed[6][:200] + "...")

청크 5의 끝부분:
...거지도록 누가 빼앗을 듯이 처박질 하더니만 그날 저녁부터 가슴이 땅긴다, 배가 켕긴다고 눈을 홉뜨고 지랄병을 하였다. 그때 김 첨지는 열화와 같이 성을 내며,
“에이, 오라질 년, 조롱복은 할 수가 없어, 못 먹어 병, 먹어서 병, 어쩌란 말이야! 왜 눈을 바루 뜨지 못해!”하고 김 첨지는 앓는 이의 뺨을 한 번 후려갈겼다. 홉

청크 6의 시작부분  <- 위 문장이 여기서 이어진다
뜬 눈은 조금 바루어졌건만 이슬이 맺히었다. 김 첨지의 눈시울도 뜨끈뜨끈하였다.
이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다.
“이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주...


In [7]:
# 문단 청킹은 줄 바꿈에서만 자르기 때문에 문장이 잘리지 않는다.
paras = chunk_paragraphs(doc, 200)

print("=" * 72)
print("문단 청크 5의 끝부분:")
print("=" * 72)
print("..." + paras[5][-200:])
print()
print("=" * 72)
print("문단 청크 6의 시작부분  <- 깔끔하게 나뉜다")
print("=" * 72)
print(paras[6][:200] + "...")

문단 청크 5의 끝부분:
...“에이, 오라질 년, 조롱복은 할 수가 없어, 못 먹어 병, 먹어서 병, 어쩌란 말이야! 왜 눈을 바루 뜨지 못해!”하고 김 첨지는 앓는 이의 뺨을 한 번 후려갈겼다. 홉뜬 눈은 조금 바루어졌건만 이슬이 맺히었다. 김 첨지의 눈시울도 뜨끈뜨끈하였다.

문단 청크 6의 시작부분  <- 깔끔하게 나뉜다
이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다.

“이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....


### 겹침(overlap)은 무엇을 해 주나

겹침은 앞 청크의 끝부분을 다음 청크의 앞에 다시 넣습니다.
그래서 반으로 잘린 문장도 **어딘가에는 온전한 형태로 존재하게** 됩니다.
대가는 중복입니다. 청크가 늘어나고, 벡터가 늘어나고, 비용이 늘어납니다.

In [8]:
plain = chunk_fixed(doc, 200, 0)
over = chunk_fixed(doc, 200, 50)
print(f"겹침 없음 : {len(plain)}개 청크")
print(f"겹침 50   : {len(over)}개 청크  (저장·임베딩 비용 +{100 * (len(over) / len(plain) - 1):.0f}%)")
print()
print("청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:")
print("---")
print(ENC.decode(ENC.encode(over[3])[-50:]))
print("---")
print(ENC.decode(ENC.encode(over[4])[:50]))

겹침 없음 : 55개 청크
겹침 50   : 73개 청크  (저장·임베딩 비용 +33%)

청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:
---
 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈
---
 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈


## 5. 임베딩하고 검색하기

벡터 데이터베이스를 쓰지 않습니다.
청크가 수백 개뿐일 때는 **작은 행렬 하나와 내적(dot product) 한 번**이면 검색이 끝납니다.
이 규모에서 DB는 오히려 군더더기입니다.

벡터를 길이 1로 정규화해 두면 내적이 곧 **코사인 유사도**가 됩니다.

In [9]:
def embed(texts, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = ai.embeddings.create(model="text-embedding-3-small",
                                 input=texts[i : i + batch])
        out.extend(d.embedding for d in r.data)
    V = np.asarray(out, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)   # 단위 벡터 -> 내적 == 코사인 유사도
    return V


INDEX = {}


def build(name, chunks):
    """한 번 임베딩한 결과를 기억해 둔다. (같은 이름은 다시 계산하지 않음)"""
    if name not in INDEX:
        print(f"임베딩 중: {name} ({len(chunks)}개 청크, 약 ${sum(map(ntok, chunks))/1e6*0.02:.4f})")
        INDEX[name] = (chunks, embed(chunks))
    return INDEX[name]


def search(query, name, k=3):
    chunks, V = INDEX[name]
    q = embed([query])[0]
    sims = V @ q
    top = np.argsort(-sims)[:k]
    return [(float(sims[i]), chunks[i]) for i in top]

In [10]:
build("fixed-200", chunk_fixed(doc, 200))
build("paragraphs-200", chunk_paragraphs(doc, 200))
print("\n준비된 인덱스:", list(INDEX))

임베딩 중: fixed-200 (55개 청크, 약 $0.0002)
임베딩 중: paragraphs-200 (57개 청크, 약 $0.0002)

준비된 인덱스: ['fixed-200', 'paragraphs-200']


## 6. 질문해 보기

점수는 참고만 하고, **돌아온 대목을 직접 읽어서** 쓸만한지 판단해 보세요.

소설은 백과사전 문서와 달리 **사실이 문장에 직접 적혀 있지 않고 묘사로 표현**됩니다.
그래서 점수가 전반적으로 낮게 나옵니다. 점수의 절대값보다 **순위**를 보세요.

In [11]:
def show(query, name, k=3, width=400):
    print("=" * 72)
    print(f"Q: {query}      [{name}]")
    print("=" * 72)
    for score, text in search(query, name, k):
        snippet = " ".join(text.split())[:width]
        print(f"\n[{score:.3f}] {snippet}...")


show("아내가 앓아누워 기침하는 대목", "paragraphs-200")

Q: 아내가 앓아누워 기침하는 대목      [paragraphs-200]

[0.457] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....

[0.394] 첫번에 삼십 전, 둘째 번에 오십 전 - 아침 댓바람에 그리 흔치 않은 일이었다. 그야말로 재수가 옴붙어서 근 열흘 동안 돈 구경도 못한 김 첨지는 십 전짜리 백통화 서 푼, 또는 다섯 푼이 찰깍하고 손바닥에 떨어질 제 거의 눈물을 흘릴 만큼 기뻤었다. 더구나 이날 이때에 이 팔십 전이라는 돈이 그에게 얼마나 유용한지 몰랐다. 컬컬한 목에 모주 한 잔도 적실 수 있거니와 그보다도 앓는 아내에게 설렁탕 한 그릇도 사다줄 수 있음이다....

[0.383] “여보게 돈 떨어졌네, 왜 돈을 막 끼얹나.” 이런 말을 하며 일변 돈을 줍는다. 김 첨지는 취한 중에도 돈의 거처를 살피는 듯이 눈을 크게 떠서 땅을 내려다보다가 불시에 제 하는 짓이 너무 더럽다는 듯이 고개를 소스라치자 더욱 성을 내며, “봐라 봐! 이 더러운 놈들아, 내가 돈이 없나, 다리뼉다구를 꺾어놓을 놈들 같으니.”하고 치삼의 주워주는 돈을 받아, “이 원수엣 돈! 이 육시를 할 돈!”하면서, 풀매질을 친다. 벽에 맞아 떨어진 돈은 다시 술 끓이는 양푼에 떨어지며 정당한 매를 맞는다는 듯이 쨍하고 울었다....


In [12]:
show("설렁탕을 사 왔는데 왜 먹지를 못하니", "paragraphs-200")

Q: 설렁탕을 사 왔는데 왜 먹지를 못하니      [paragraphs-200]

[0.645] “설렁탕을 사다놓았는데 왜 먹지를 못하니, 왜 먹지를 못하니... 괴상하게도 오늘은! 운수가 좋더니만... ”...

[0.526] 이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다. “이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....

[0.443] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....


In [13]:
# 같은 내용을 다르게 물어보면? -- 점수와 순위를 잘 보세요.
show("김 첨지의 아내는 어떤 상태인가?", "paragraphs-200")

Q: 김 첨지의 아내는 어떤 상태인가?      [paragraphs-200]

[0.583] 김 첨지의 눈은 벌써 개개 풀리기 시작하였다. 석쇠에 얹힌 떡 두 개를 숭덩숭덩 썰어서 볼을 불룩거리며 또 곱배기 두 잔을 부어라 하였다. 치삼은 의아한 듯이 김 첨지를 보며, “여보게 또 붓다니, 벌써 우리가 넉 잔씩 먹었네, 돈이 사십 전일세.”라고 주의시켰다....

[0.528] 치삼의 끄는 손을 뿌리치더니 김 첨지는 눈물이 글썽글썽한 눈으로 싱그레 웃는다. “죽기는 누가 죽어.”하고 득의가 양양. “죽기는 왜 죽어, 생때같이 살아만 있단다. 그 오라질 년이 밥을 죽이지. 인제 나한테 속았다.”하고 어린애 모양으로 손뼉을 치며 웃는다....

[0.513] 김 첨지는 취중에도 설렁탕을 사가지고 집에 다달았다. 집이라 해도 물론 셋집이요, 또 집 전체를 세든 게 아니라 안과 뚝떨어진 행랑방 한 간을 빌려 든 것인데 물을 길어대고 한 달에 일 원씩 내는 터이다. 만일 김 첨지가 주기를 띠지 않았던들 한 발을 대문에 들여놓았을 제 그곳을 지배하는 무시무시한 정적(靜寂) — 폭풍우가 지나간 뒤의 바다 같은 정적에 다리가 떨렸으리라....


### 점수가 더 높은데 틀렸습니다 — 꼭 읽어 주세요

세 질문의 결과를 나란히 놓고 보세요.

| 질문 | 1위 점수 | 1위로 돌아온 대목 | 맞았나? |
|---|---|---|---|
| 아내가 앓아누워 기침하는 대목 | 0.457 | "그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다…" | ✅ |
| 설렁탕을 사 왔는데 왜 먹지를 못하니 | 0.645 | 소설의 마지막 대사 | ✅ |
| **김 첨지의 아내는 어떤 상태인가?** | **0.583** | 김 첨지가 술집에서 떡과 술을 먹는 장면 | ❌ |

세 번째 질문은 첫 번째 질문과 **묻는 내용이 사실상 같습니다.**
그런데 점수는 0.583으로 더 높은데, 돌아온 대목은 아내와 아무 상관없는 술집 장면입니다.

**왜 이런 일이 생길까요?**

질문에 `김 첨지`라는 인물 이름이 들어가자, **인물 이름과 대화가 빽빽하게 들어 있는 대목**이
의미상 더 가깝게 계산되었습니다. 소설은 백과사전과 달리 *"아내는 폐병을 앓고 있다"* 처럼
사실을 직접 진술하지 않고 **묘사로 돌려 말하기** 때문에, "어떤 상태인가" 같은 추상적인 질문은
특정 대목에 잘 달라붙지 않습니다.

**여기서 얻을 교훈 두 가지입니다.**

1. **점수는 질문끼리 비교할 수 없습니다.** 0.583이 0.457보다 크다고 해서 더 좋은 검색이 아닙니다.
   점수는 **같은 질문 안에서 순위를 매기는 용도**로만 의미가 있습니다.
2. **질문을 쓰는 방식이 검색 품질을 바꿉니다.** 같은 것을 물어도
   "~은 어떤 상태인가"(추상적)보다 "~하는 대목"(장면 묘사)이 훨씬 잘 찾습니다.
   실무에서는 사용자의 질문을 LLM으로 검색에 적합한 문장으로 다시 쓰는
   **질의 재작성(query rewriting)** 을 쓰기도 합니다.

## 7. 실험: 청크 크기가 결과를 바꾸는가?

같은 문서, 같은 질문, 같은 임베딩 모델입니다. **청크 크기만** 바뀝니다.
결과를 읽고, 어느 쪽을 LLM에게 컨텍스트로 넘기고 싶은지 직접 판단해 보세요.

작은 청크는 정확하지만 주변 맥락이 잘립니다.
큰 청크는 맥락이 충분하지만 정작 답이 관계없는 내용에 파묻힙니다.

In [14]:
QUESTION = "아내가 앓아누워 기침하는 대목"

for size in [100, 200, 400, 800]:
    name = f"para-{size}"
    build(name, chunk_paragraphs(doc, size))
    score, text = search(QUESTION, name, k=1)[0]
    snippet = " ".join(text.split())
    print("=" * 72)
    print(f"청크 크기 {size:>4}토큰  |  {len(INDEX[name][0]):>3}개 청크  |  "
          f"1위 점수 {score:.3f}  |  이 청크는 {ntok(text)}토큰")
    print("=" * 72)
    print(snippet[:600] + ("..." if len(snippet) > 600 else ""))
    print()

임베딩 중: para-100 (76개 청크, 약 $0.0002)
청크 크기  100토큰  |   76개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.

임베딩 중: para-200 (57개 청크, 약 $0.0002)
청크 크기  200토큰  |   57개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.

임베딩 중: para-400 (36개 청크, 약 $0.0002)
청크 크기  400토큰  |   36개 청크  |  1위 점수 0.457  |  이 청크는 277토큰
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 

### 방금 무슨 일이 일어났나 — 넘어가기 전에 읽어 주세요

| 청크 크기 | 청크 개수 | 1위 점수 | 돌아온 청크 |
|---|---|---|---|
| 100 | 76개 | 0.457 | 아내의 병을 묘사한 문단 (277토큰) |
| 200 | 57개 | 0.457 | **위와 완전히 같은 청크** |
| 400 | 36개 | 0.457 | **위와 완전히 같은 청크** |
| 800 | 17개 | 0.425 | 같은 문단으로 시작하지만 뒤에 다른 장면들이 줄줄이 붙은 788토큰 덩어리 |

**100, 200, 400이 왜 똑같을까요?**

`chunk_paragraphs`는 문단을 **쪼개지 않고 모으기만** 합니다.
`chunk_size`는 "이 크기로 자른다"가 아니라 **"이 크기를 넘지 않게 문단을 합친다"는 상한**입니다.
우리가 찾는 대목은 그 자체로 **277토큰짜리 문단 하나**입니다.
상한을 100으로 낮춰도 이 문단은 쪼개지지 않고 그대로 남습니다. 목표 200을 넘는데도 그렇습니다.

> 여기서 얻을 교훈: **`chunk_size`를 줄였는데 결과가 그대로라면 청커가 무시하고 있는 것입니다.**
> 숫자만 바꾸지 말고, 청크가 몇 개 만들어졌는지(76 → 57 → 36 → 17개)와
> 실제로 돌아온 청크의 크기를 함께 확인하세요.

**800에서는 왜 점수가 떨어졌을까요?**

여러 문단이 하나로 합쳐지면서 아내의 병 묘사가 **다른 장면들에 희석**되었습니다.
임베딩은 텍스트 전체의 평균에 가깝기 때문에, 다른 내용이 섞일수록 질문과의 유사도가 낮아집니다(0.457 → 0.425).
답이 청크 안에 들어 있긴 하지만, LLM에게 넘길 컨텍스트로는 788토큰 중 대부분이 군더더기입니다.

**소설은 백과사전 문서와 다릅니다.**

`chunk_basic_ko1.ipynb`(위키백과 「인공지능」)에서는 문단 하나가 1,295토큰까지 나왔습니다.
반면 이 소설은 최대 345토큰으로 **문단 길이가 고른 편**입니다.
대화가 많아 줄이 자주 바뀌기 때문인데, 덕분에 문단 청킹만으로도 크기가 안정적입니다.
**글의 종류에 따라 알맞은 청킹 전략이 다르다**는 뜻입니다.

### 고정 크기 vs 문단 경계

같은 질문, 같은 목표 크기입니다.
차이는 **문장 경계를 지키느냐 마느냐** 하나뿐입니다.

In [15]:
for name in ["fixed-200", "paragraphs-200"]:
    score, text = search(QUESTION, name, k=1)[0]
    print("=" * 72)
    print(f"{name}   점수 {score:.3f}")
    print("=" * 72)
    print(" ".join(text.split())[:500])
    print()

fixed-200   점수 0.403
�하고 손바닥에 떨어질 제 거의 눈물을 흘릴 만큼 기뻤었다. 더구나 이날 이때에 이 팔십 전이라는 돈이 그에게 얼마나 유용한지 몰랐다. 컬컬한 목에 모주 한 잔도 적실 수 있거니와 그보다도 앓는 아내에게 설렁탕 한 그릇도 사다줄 수 있음이다. 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니

paragraphs-200   점수 0.457
그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다.



### 결과: 문단 경계를 지킨 쪽이 이겼습니다

| | 1위 점수 | 돌아온 내용 |
|---|---|---|
| `fixed-200` | 0.403 | **깨진 글자(`�`)로 시작**합니다. 앞부분은 돈 이야기이고, 아내의 병 묘사는 뒤에 일부만 걸쳐 있습니다 |
| `paragraphs-200` | **0.457** | 아내의 병을 묘사한 문단이 처음부터 끝까지 온전히 |

**맨 앞의 깨진 글자를 보세요.** 한글은 글자 하나가 여러 토큰으로 쪼개지기 때문에,
토큰 개수만 세어 자르면 **글자 한가운데가 잘려 나갑니다.**
영어에서는 단어가 잘리는 정도지만, 한글에서는 **글자 자체가 깨집니다.**

잘린 청크를 LLM에게 컨텍스트로 넘기면 답변 품질도 함께 떨어집니다.
**문장 경계를 지키는 것만으로 검색 품질이 올라갑니다.**

## 8. 직접 해 보기

질문을 바꿔 가며 다시 실행해 보세요. 시도해 볼 만한 것들입니다.

- **작품 전체에 걸친 질문** (*"이 소설의 제목이 왜 반어적인가?"*)
  — 청크 하나에 답이 들어 있을 수 있을까요? 임베딩 검색의 한계가 드러납니다.
- **유명한 마지막 대사** (*"설렁탕을 사 왔는데 왜 먹지를 못하니"*)
  — 구체적인 문장은 잘 찾아냅니다.
- **작품에 없는 내용** (*"파이썬으로 웹 서버 만드는 법은?"*)
  — 유사도 점수를 보세요. 임베딩 검색은 **항상 무언가를 돌려줍니다.**
    관련 내용이 없다는 것을 알려 주는 유일한 단서가 점수입니다.

In [16]:
MY_QUESTION = "설렁탕을 사 왔는데 왜 먹지를 못하니"

show(MY_QUESTION, "paragraphs-200", k=3)

Q: 설렁탕을 사 왔는데 왜 먹지를 못하니      [paragraphs-200]

[0.645] “설렁탕을 사다놓았는데 왜 먹지를 못하니, 왜 먹지를 못하니... 괴상하게도 오늘은! 운수가 좋더니만... ”...

[0.526] 이 환자가 그러고도 먹는 데는 물리지 않았다. 사흘 전부터 설렁탕 국물이 마시고 싶다고 남편을 졸랐다. “이런 오라질 년! 조밥도 못 먹는 년이 설렁탕은, 또 처먹고 지랄병을 하게.”라고, 야단을 쳐보았건만, 못 사주는 마음이 시원치는 않았다....

[0.443] 그의 아내가 기침으로 쿨럭거리기는 벌써 달포가 넘었다. 조밥도 굶기를 먹다시피 하는 형편이니 물론 약 한 첩 써 본 일이 없다. 구태여 쓰려면 못 쓸 바도 아니로되 그는 병이란 놈에게 약을 주어 보내면 재미를 붙여서 자꾸 온다는 자기의 신조(信條)에 어디까지 충실하였다. 따라서 의사에게 보인 적이 없으니 무슨 병인지는 알 수 없으되 반듯이 누워 가지고, 일어나기는새로 모로도 못 눕는 걸 보면 중증은 중증인 듯. 병이 이대도록 심해지기는 열흘 전에 조밥을 먹고 체한 때문이다....


In [17]:
# 작품에 답이 없는 질문 -- 내용보다 점수를 보세요.
show("파이썬으로 웹 서버 만드는 법은?", "paragraphs-200", k=3)

Q: 파이썬으로 웹 서버 만드는 법은?      [paragraphs-200]

[0.166] “젠장맞을 것! 이 비를 맞으며 빈 인력거를 털털거리고 돌아를 간담. 이런 빌어먹을, 제 할미를 붙을 비가 왜 남의 상판을 딱딱 때려!”...

[0.116] 그는 불행에 다닥치기 전 시간을 얼마쯤이라도 늘리려고 버르적거렸다. 기적(奇蹟)에 가까운 벌이를 하였다는 기쁨을 할 수 있으면 오래 지니고 싶었다. 그는 두리번두리번 사면을 살피었다. 그 모양은 마치 자기 집 — 곧 불행을 향하고 달려가는 제 다리를 제 힘으로는 도저히 어찌할 수 없으니 누구든지 나를 좀 잡아 다고, 구해 다고 하는 듯하였다....

[0.116] 그는 몹시 홧증을 내며 누구에게 반항이나 하는 듯이 게걸거렸다. 그럴 즈음에 그의 머리엔 또 새로운 광명이 비쳤나니 그것은 <이러구 갈 게 아니라 이 근처를 빙빙 돌며 차 오기를 기다리면 또 손님을 태우게 될는지도 몰라>란 생각이었다. 오늘 운수가 괴상하게도 좋으니까 그런 요행이 또한번 없으리라고 누가 보증하랴. 꼬리를 굴리는 행운이 꼭 자기를 기다리고 있다고 내기를 해도 좋을 만한 믿음을 얻게 되었다. 그렇다고 정거장 인력거꾼의 등살이 무서우니 정거장 앞에 섰을 수는 없었다....


> **점수를 비교해 보세요.** 작품에 답이 있는 질문은 0.45~0.65가 나왔는데,
> 파이썬 질문은 **0.166, 0.116, 0.116** 입니다.
> 그런데도 검색은 **빈손으로 돌아오지 않습니다.** 무조건 세 개를 가져옵니다.
>
> 실무에서는 이 점수에 **하한선(threshold)** 을 두어, 기준보다 낮으면
> "관련 내용을 찾지 못했습니다"라고 처리합니다.
> 그러지 않으면 LLM이 엉뚱한 대목을 근거로 그럴듯한 답을 지어냅니다.
>
> 다만 하한선은 **문서 종류마다 달라야 합니다.** 이 소설에서는 정답이 0.45 정도인데,
> `chunk_basic_ko1.ipynb`의 백과사전 문서에서는 정답이 0.5~0.7이었습니다.
> 같은 0.45를 백과사전에서는 "관련 없음"으로 봐야 할 수도 있습니다.

## 9. 벡터 DB에 저장하기 (Chroma)

지금까지는 임베딩을 **numpy 행렬**에 담아 두고 내적으로 검색했습니다.
청크가 57개뿐이라 그것으로 충분했지만, 실제 서비스에서는 두 가지가 아쉽습니다.

1. **커널을 껐다 켜면 사라집니다.** 매번 다시 임베딩해야 하고, 그때마다 비용이 듭니다.
2. **문서가 많아지면 느려집니다.** 전체를 훑는 방식이라 청크 수에 비례해 느려집니다.

**벡터 DB**는 벡터를 디스크에 저장하고, 근사 최근접 탐색(ANN) 색인으로 빠르게 찾아 줍니다.
여기서는 파일로 저장되는 가벼운 DB인 **Chroma**를 씁니다.

두 청킹 방식을 **각각 별도의 컬렉션**으로 저장해서, 같은 질문을 두 컬렉션에 던져 비교할 것입니다.

| 컬렉션 | 내용 |
|---|---|
| `fixed-200` | 토큰 200개마다 기계적으로 자른 청크 |
| `paragraphs-200` | 문단 경계를 지켜 자른 청크 |

> 이미 5번에서 임베딩을 계산해 `INDEX`에 담아 두었습니다.
> 그 벡터를 그대로 DB에 넣으므로 **추가 임베딩 비용은 0원**입니다.

> ⚠️ **폴더를 직접 지우면 안 됩니다.**
> 실습을 반복하려고 `shutil.rmtree("chroma_chunk_ko2")`로 폴더를 통째로 지우면
> 다음과 같은 오류가 납니다.
>
> ```
> InternalError: Collection [fixed-200] already exists
> InternalError: Error creating hnsw segment reader: Nothing found on disk
> ```
>
> 커널에 살아 있는 Chroma 클라이언트가 **"컬렉션이 있다"는 정보를 메모리에 들고 있는데**
> 디스크의 파일만 사라져서, 앞뒤가 맞지 않는 상태가 되기 때문입니다.
> 폴더가 아니라 **`client.delete_collection(name)`으로 컬렉션을 지워야** 합니다.
> (커널을 재시작한 뒤라면 폴더를 지워도 괜찮습니다.)

In [18]:
import chromadb
from chromadb.errors import NotFoundError

DB_PATH = "chroma_chunk_ko2"

client = chromadb.PersistentClient(path=DB_PATH)

collections = {}
for name in ["fixed-200", "paragraphs-200"]:
    chunks, V = INDEX[name]          # 5번에서 만들어 둔 청크와 임베딩 벡터

    # 같은 이름의 컬렉션이 이미 있으면 지우고 새로 만든다. (이 셀을 몇 번이든 다시 실행할 수 있도록)
    try:
        client.delete_collection(name)
        print(f"{name:16s} 기존 컬렉션 삭제")
    except NotFoundError:
        pass                          # 없으면 그냥 새로 만들면 된다

    col = client.create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"},   # 코사인 거리로 비교
    )
    col.add(
        ids=[f"{name}-{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=V.tolist(),               # 이미 계산한 벡터를 재사용 (재임베딩 안 함)
        metadatas=[{"chunk_id": i, "tokens": ntok(c)} for i, c in enumerate(chunks)],
    )
    collections[name] = col
    print(f"{name:16s} {col.count():>3}개 청크 저장")

print(f"\n저장 위치: {DB_PATH}/")

fixed-200        기존 컬렉션 삭제
fixed-200         55개 청크 저장
paragraphs-200   기존 컬렉션 삭제
paragraphs-200    57개 청크 저장

저장 위치: chroma_chunk_ko2/


### DB에서 검색하기

Chroma는 유사도가 아니라 **거리(distance)** 를 돌려줍니다.
벡터를 길이 1로 정규화해 두었으므로 `유사도 = 1 - 거리` 로 환산하면
5번에서 numpy로 계산한 값과 같아집니다.

In [19]:
def search_db(query, name, k=3):
    """벡터 DB에서 검색한다. (점수, 본문, 청크번호, 토큰수) 리스트를 돌려준다."""
    qv = embed([query])[0]                       # 질문을 벡터로
    res = collections[name].query(query_embeddings=[qv.tolist()], n_results=k)
    return [
        (1 - dist, text, meta["chunk_id"], meta["tokens"])
        for dist, text, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0])
    ]


# numpy 방식과 결과가 같은지 확인
q = "아내가 앓아누워 기침하는 대목"
print("numpy :", f"{search(q, 'paragraphs-200', k=1)[0][0]:.3f}")
print("chroma:", f"{search_db(q, 'paragraphs-200', k=1)[0][0]:.3f}")

numpy : 0.457
chroma: 0.457


## 10. Gradio로 두 청킹 방식 비교하기

질문을 입력하면 **두 컬렉션에 같은 질문을 던져** 결과를 나란히 보여 주고,
무엇이 어떻게 다른지 **문장으로 정리해 주는** 화면을 만듭니다.

비교 문장은 세 가지를 자동으로 판단합니다.

1. **점수** — 어느 쪽 1위 점수가 높은가
2. **같은 대목인가** — 두 방식이 찾아온 본문이 얼마나 겹치는가
3. **잘렸는가** — 청크가 문장 중간에서 끊겼는지, 한글 글자가 깨졌는지

In [20]:
from difflib import SequenceMatcher


def diagnose(text):
    """청크가 잘렸는지 진단한다."""
    problems = []
    if "\ufffd" in text[:5]:
        problems.append("글자가 깨진 채 시작")
    if not text.rstrip().endswith((".", "!", "?", "”", "\"", "…", "-")):
        problems.append("문장 중간에서 끝남")
    return problems


def build_summary(query, fixed, paras):
    """두 결과를 비교하는 문장을 만든다."""
    fs, ft = fixed[0][0], fixed[0][1]
    ps, pt = paras[0][0], paras[0][1]

    lines = [f"### 「{query}」 비교 결과", ""]

    # 1) 점수
    winner = "문단 청킹" if ps > fs else "고정 크기 청킹"
    lines.append(f"- **점수**: 고정 크기 `{fs:.3f}` vs 문단 `{ps:.3f}` "
                 f"→ **{winner}**이 `{abs(ps - fs):.3f}` 높습니다.")

    # 2) 같은 대목을 찾았는가
    overlap = SequenceMatcher(None, ft, pt).ratio()
    if overlap > 0.3:
        lines.append(f"- **내용**: 두 방식이 **같은 대목**을 찾았습니다 (본문 일치도 {overlap:.0%}). "
                     f"어디서 잘랐는지만 다릅니다.")
    else:
        lines.append(f"- **내용**: 두 방식이 **서로 다른 대목**을 가져왔습니다 (본문 일치도 {overlap:.0%}). "
                     f"직접 읽고 어느 쪽이 질문에 맞는지 판단해 보세요.")

    # 3) 잘림 진단
    fp, pp = diagnose(ft), diagnose(pt)
    if fp:
        lines.append(f"- **잘림**: 고정 크기 1위 청크는 {', '.join(fp)}. "
                     f"한글은 글자 하나가 여러 토큰이라 토큰 경계로 자르면 글자가 깨질 수 있습니다.")
    if not pp:
        lines.append("- **잘림**: 문단 청크는 문장이 온전합니다.")
    elif pp:
        lines.append(f"- **잘림**: 문단 1위 청크도 {', '.join(pp)} (대화체라 따옴표로 끝나는 경우가 있습니다).")

    # 4) 크기
    lines.append(f"- **크기**: 고정 크기 {fixed[0][3]}토큰 / 문단 {paras[0][3]}토큰")
    return "\n".join(lines)


def format_results(title, results):
    out = [f"### {title}", ""]
    for rank, (score, text, chunk_id, tokens) in enumerate(results, 1):
        body = " ".join(text.split())
        out.append(f"**{rank}위 · 점수 {score:.3f} · 청크 {chunk_id} · {tokens}토큰**")
        out.append("")
        out.append(f"> {body[:400]}{'...' if len(body) > 400 else ''}")
        out.append("")
    return "\n".join(out)


def compare(query, k):
    if not query.strip():
        return "질문을 입력하세요.", "", ""
    k = int(k)
    fixed = search_db(query, "fixed-200", k)
    paras = search_db(query, "paragraphs-200", k)
    return (build_summary(query, fixed, paras),
            format_results("고정 크기 청킹 (fixed-200)", fixed),
            format_results("문단 청킹 (paragraphs-200)", paras))


# UI 없이 함수만 먼저 확인
summary, _, _ = compare("아내가 앓아누워 기침하는 대목", 3)
print(summary)

### 「아내가 앓아누워 기침하는 대목」 비교 결과

- **점수**: 고정 크기 `0.403` vs 문단 `0.457` → **문단 청킹**이 `0.054` 높습니다.
- **내용**: 두 방식이 **서로 다른 대목**을 가져왔습니다 (본문 일치도 22%). 직접 읽고 어느 쪽이 질문에 맞는지 판단해 보세요.
- **잘림**: 고정 크기 1위 청크는 글자가 깨진 채 시작, 문장 중간에서 끝남. 한글은 글자 하나가 여러 토큰이라 토큰 경계로 자르면 글자가 깨질 수 있습니다.
- **잘림**: 문단 청크는 문장이 온전합니다.
- **크기**: 고정 크기 200토큰 / 문단 277토큰


In [21]:
import gradio as gr

with gr.Blocks(title="청킹 방식 비교 검색") as demo:
    gr.Markdown("# 청킹 방식 비교 검색\n"
                "「운수 좋은 날」에서 같은 질문을 **두 가지 청킹 방식**으로 검색하고 결과를 비교합니다.")

    with gr.Row():
        question = gr.Textbox(label="질문", value="아내가 앓아누워 기침하는 대목", scale=4)
        k = gr.Slider(1, 5, value=3, step=1, label="가져올 청크 수 (k)", scale=1)

    btn = gr.Button("검색", variant="primary")

    gr.Examples(
        examples=[
            ["설렁탕을 사 왔는데 왜 먹지를 못하니"],
            ["김 첨지의 아내는 어떤 상태인가?"],
            ["김 첨지가 친구 치삼과 술을 마시는 장면"],
            ["비 오는 날씨 묘사"],
            ["파이썬으로 웹 서버 만드는 법은?"],
        ],
        inputs=question,
    )

    summary_out = gr.Markdown()
    with gr.Row():
        fixed_out = gr.Markdown()
        paras_out = gr.Markdown()

    outputs = [summary_out, fixed_out, paras_out]
    btn.click(compare, inputs=[question, k], outputs=outputs)
    question.submit(compare, inputs=[question, k], outputs=outputs)

demo.launch(inbrowser=True)

d:\ai_edu_main\llm_edu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [22]:
demo.close()

Closing server running on port: 7860


### 화면에서 확인해 볼 것

- **예시 질문을 차례로 눌러 보세요.** 질문 종류에 따라 승자가 바뀝니다.
- **`k`를 1로 줄여 보세요.** 1위만 봤을 때 답을 놓치는 경우가 생깁니다.
- **"파이썬으로 웹 서버 만드는 법은?"** 처럼 작품에 없는 질문을 넣어 보세요.
  점수가 0.2 아래로 떨어지지만, 두 방식 모두 **무언가를 돌려준다**는 점을 확인할 수 있습니다.
- **같은 내용을 다르게 물어보세요.** ("아내가 앓아누워 기침하는 대목" vs "김 첨지의 아내는 어떤 상태인가?")
  6번에서 본 것처럼, 질문 표현만 바꿔도 결과가 완전히 달라집니다.

> 이 화면은 **검색 단계까지만** 보여 줍니다.
> 찾아온 청크를 LLM에게 넘겨 답변을 생성하는 전체 RAG 흐름은 `rag_basic_ko.ipynb`에 있습니다.

## 다음 단계

- **다른 작품으로 바꿔 보세요.** `TITLE`만 바꾸면 됩니다.
  위키문헌에는 저작권이 만료된 한국 근대문학이 많습니다.
  예: `"메밀꽃 필 무렵"`(이효석), `"날개"`(이상), `"벙어리 삼룡이"`(나도향)
  ```python
  # 제목이 정확해야 합니다. 없는 제목이면 'extract' 키가 없어 KeyError가 납니다.
  # 검색: https://ko.wikisource.org 에서 작품명을 찾아보세요.
  ```
- **백과사전 문서와 비교해 보세요.** `chunk_basic_ko1.ipynb`는 같은 코드를
  한국어 위키백과 「인공지능」 문서로 돌립니다.
  소설과 설명문에서 **검색 점수와 청크 모양이 어떻게 다른지** 비교해 보면 좋습니다.
- **문서 여러 개를 다루기** — 문서가 많아지면 "어느 청크인가" 이전에
  "어느 문서인가"를 먼저 고르는 문제가 생깁니다.
- **눈으로 보지 말고 측정하기** — `rag_chunk_embed.md`와 `science_qa.ipynb`가
  정답 라벨이 있는 데이터로 이 작업을 제대로 합니다.
  핵심 아이디어는, 청크 개수를 고정하지 말고 **컨텍스트 총량(토큰 예산)을 고정**해서
  비교해야 한다는 것입니다.
- **LLM까지 연결하려면** — 이 노트북은 검색까지만 다룹니다.
  찾아온 청크를 LLM에게 넘겨 답변을 만드는 전체 RAG 흐름은 `rag_basic_ko.ipynb`에 있습니다.